# Falcon-H1-3B cyber QLoRA — notebook driver

This notebook is a **thin client over `src/cyberft`**. Nothing is implemented here: every
cell calls the same library function the CLI and the tests call, so what you see running
is what ships.

| Cell | Library call | File |
|---|---|---|
| config | `load_config`, `amp_dtype` | `src/cyberft/config.py`, `model.py` |
| data | `prepare_data`, `data_manifest` | `src/cyberft/data.py` |
| smoke train | `train(cfg, smoke=True)` | `src/cyberft/train.py` |
| full train | `train(cfg)` | `src/cyberft/train.py` |
| evaluate | `evaluate` → `EvalReport` | `src/cyberft/evaluate.py` |
| generate | `generate` | `src/cyberft/evaluate.py` |
| export | `export_gguf` | `src/cyberft/export.py` |

**Hardware:** bf16 on Ampere+ (fast, fused Mamba kernels available); fp16 on a T4
(works, ~4-5x slower). `amp_dtype` picks automatically.

## 0. Bootstrap

Find the repo root, put `src/` on the path (so this works without installing), and
configure logging once. The library never configures logging or exits the process itself.

In [1]:
!pip install -e ".[train]"

Obtaining file:///content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 33.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 57.4 MB/s  0:00:006m0:00:0100:01
  Building editable for cyberft (pyproject.toml) ... done
  Created wheel for cyberft: filename=cyberft-0.3.2-0.editable-py3-none-any.whl size=1414 sha256=fed466c816059c67ae6a8164503191ab820feec3003b919d8033c4a6e8c43369
  Stored in directory: /tmp/pip-ephem-wheel-cache-8qxa_1ve/wheels/04/1c/82/119baa7649e8aa11e93bf2a5ccf84b70ae90a4652cbe18aa22
Successfully built cyberft
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully unin

In [2]:
import logging
import sys
from pathlib import Path

REPO = Path.cwd()
print(REPO)
while not (REPO / "configs" / "qlora-3b.yaml").exists() and REPO.parent != REPO:
    REPO = REPO.parent
if not (REPO / "configs" / "qlora-3b.yaml").exists():
    raise RuntimeError("open this notebook from inside falcon-cyber-ft/")

sys.path.insert(0, str(REPO / "src"))          # no install needed
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s")
print("repo:", REPO)

/content
repo: /content


## 1. Configuration and hardware

`load_config` validates the YAML — including the two requirements that are easy to get
wrong: Mamba's `out_proj` must stay unquantized, and it must never be a LoRA target.

In [3]:
import torch

import cyberft
from cyberft import amp_dtype, load_config

cfg = load_config(REPO / "configs" / "qlora-3b.yaml")

print(f"cyberft   {cyberft.__version__}")
print(f"gpu       {torch.cuda.get_device_name(0)}")
print(f"dtype     {amp_dtype(cfg)}")
print(f"model     {cfg.model.id}")
print(f"lora      r={cfg.lora.r} alpha={cfg.lora.alpha} targets={list(cfg.lora.target_modules)}")
print(f"skip-bnb  {list(cfg.model.bnb_skip_modules)}")
print(f"train     {cfg.train.micro_batch} x {cfg.train.grad_accum} accum, "
      f"lr={cfg.train.lr}, {cfg.train.epochs} epoch(s)")

print()


cyberft   0.3.2
gpu       Tesla T4 (SM 7.x)
dtype     torch.float16
model     tiiuae/Falcon-H1-3B-Instruct
lora      r=32 alpha=16 targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'gate_proj', 'up_proj', 'down_proj']
skip-bnb  ['out_proj']
train     1 x 8 accum, lr=0.0002, 1 epoch(s)



## 2. Corpus

Built once and cached under `data/processed/{smoke,full}` with a `manifest.json`
(row counts, token percentiles, content hash). `force=True` rebuilds from the sources.

In [7]:
from cyberft import data_manifest, prepare_data

SMOKE = True        # 400 rows; set False for the full corpus
REBUILD = False     # True re-downloads and re-renders everything

splits = prepare_data(cfg, smoke=SMOKE, force=REBUILD)
print({name: len(ds) for name, ds in splits.items()})
print(data_manifest(cfg, smoke=SMOKE))

{'train': 322, 'eval': 32}
{'fingerprint': 'a0d8c5a0369ae62b', 'content_sha256': '73b7f8c5e71ecaf26ff698bc7025c328265a219f75f85b6d062109f0f5329b73', 'model_id': 'tiiuae/Falcon-H1-3B-Instruct', 'sources': {'redteam': 'WNT3D/Ultimate-Offensive-Red-Team', 'kali': 'suryanshp1/kali-linux-pentesting-data', 'kimi': 'echel0nn1881/kimi-cyber-reasoning', 'cti': 'reloading0101/threat-intelligence-dataset', 'exploitdb': 'Waiper/ExploitDB_DataSet', 'linuxcmd': 'Romit2004/LinuxCommands', 'debian': '2etatg/Debian_Hacking_Networking'}, 'revisions': {'redteam': '1aabece7216f0f7fe531c43d89fe846116dc47c2', 'kali': '91494471b90e8f8d5017881be082abf86e90544d', 'kimi': 'ab9e1244bdbb5d74a6083e78259db3b64a2c94cb', 'cti': '86eef0c28d0ef6d2edeadb4ce07c8ea67ad97be2', 'exploitdb': '3fc5a9bfc81bc9aa4c2bd9c22ce57efca05f41f0', 'linuxcmd': '2941f9386e1a8638ae0c1ce3d857862696b584d8', 'debian': '8752d19e736b09bcd6f0d717112341bc9aff316e'}, 'max_len': 2048, 'eval_max_len': 1024, 'rows': {'train': 322, 'eval': 32}, 'tokens

: 

## 3. Smoke train

64 rows, 5 steps. This is the test that would have caught every dtype failure in the
notebook version of this project before spending 20 minutes on a real run.

In [8]:
!pip install "bitsandbytes>=0.50.2"


: 

In [ ]:
from cyberft import train, train_dir

trainer = train(cfg, smoke=True)
adapter = train_dir(cfg, smoke=True)     # adapter + tokenizer + checkpoints
print("adapter:", adapter)

## 4. Full train

Guarded so "Run All" does not launch a multi-hour job. `train.resume: true` in the config
picks up the newest checkpoint automatically.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    trainer = train(cfg)
    adapter = train_dir(cfg)
    SMOKE = False          # evaluate/generate from now on use the full-data artifacts
    print("adapter:", adapter)
else:
    print("skipped; set RUN_FULL = True to train on the full corpus")

: 

## 5. Evaluate

`EvalReport` carries one loss per row. `assert_finite()` raises instead of reporting a
bare `nan`, and the summary names the worst rows with their text.

In [ ]:
from cyberft import evaluate

report = evaluate(cfg, smoke=SMOKE)
print(report.summary())
report.assert_finite()

: 

## 6. Generate

In [ ]:
from cyberft import generate

PROMPTS = [
    "Write a tcpdump command that captures only DNS traffic on eth0.",
    "How do I enumerate SMB shares on an authorised lab network?",
    "Summarise MITRE ATT&CK technique T1059.004 (Unix Shell).",
]

ANSWERS = generate(cfg, PROMPTS, adapter=adapter, max_new_tokens=256)

for question, answer in zip(PROMPTS, ANSWERS, strict=True):
    print(f"Q: {question}\nA: {answer}\n{'-' * 78}")

: 

## 7. Export GGUF

Merge → F16 → quantize. Requires llama.cpp cloned into `vendor/llama.cpp`; the function
raises `DependencyError` rather than producing a half-empty output directory.

In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    from cyberft import export_gguf
    path = export_gguf(cfg, adapter=adapter)
    print(path, f"{path.stat().st_size / 1e9:.2f} GB")
else:
    print("skipped; set RUN_EXPORT = True")

: 

## 8. Where the code lives

Everything above is defined in these files. After editing them in a live kernel, run the
last line so the notebook picks up the change.

In [ ]:
import inspect

import cyberft
import cyberft.config
import cyberft.data
import cyberft.evaluate
import cyberft.export
import cyberft.model
import cyberft.train

for module in (cyberft.config, cyberft.data, cyberft.model, cyberft.train,
               cyberft.evaluate, cyberft.export):
    print(f"{module.__name__:<20} {inspect.getsourcefile(module)}")

# after editing src/:
# import importlib; importlib.reload(cyberft)

: 

## Troubleshooting

| Symptom | Meaning |
|---|---|
| `NotBuiltError: ... - train first` | run the train cell before evaluate/export |
| `NotBuiltError: data/processed/...` | run the data cell |
| training is slow (~1.7 s/sample) | pre-Ampere GPU: fp16 + PyTorch Mamba fallback |
| `NonFiniteLossError` | `report.summary()` names the offending rows |